In [ ]:
import sys
sys.path.append('src')

import warnings
warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error()



Загрузка и подготовка данных

In [8]:
from data_utils import load_and_clean, split_data, build_vocab

data = load_and_clean("data/raw_dataset.txt")
train_df, val_df, test_df = split_data(data)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(train_df["text"].head(3))


Train: 1277468, Val: 159683, Test: 159684
436600     ok time for the pub and some food i logged ko ...
155409     it feels like strep throatbut i sure hope its ...
1594109                                   hand up in the air
Name: text, dtype: object


Загрузка обученной LTSM

In [9]:
import torch
from lstm_model import SimpleRNN
from data_utils import build_vocab
from next_token_dataset import NextTokenDataset

vocab = build_vocab(train_df)
vocab_size = len(vocab)

model = SimpleRNN(vocab_size, embedding_dim=128, hidden_size=128, output_size=vocab_size)
model.load_state_dict(torch.load("models/lstm_model.pt", map_location="cpu"))
model.eval()
print("Модель загружена")


Модель загружена


Подсчет ROUGE для LSTM

In [10]:
from eval_lstm import evaluate_model

val_dataset = NextTokenDataset(val_df, vocab)
lstm_results = evaluate_model(model, val_dataset, vocab)
print(f"LSTM ROUGE-1: {lstm_results['rouge1']:.4f}")
print(f"LSTM ROUGE-2: {lstm_results['rouge2']:.4f}")


LSTM ROUGE-1: 0.0783
LSTM ROUGE-2: 0.0082


Примеры предсказаний LTSM

In [11]:
idx2word = {v: k for k, v in vocab.items()}

print("=== Примеры предсказаний LSTM ===\n")
for x, y in list(val_dataset)[:5]:
    n = len(x)
    cut = int(n * 0.75)
    prefix = x[:cut]
    target = x[cut:]
    
    input_ids = prefix.unsqueeze(0)
    lengths = torch.tensor([len(prefix)])
    generated = model.generate(input_ids, lengths, max_new_tokens=5)
    new_tokens = generated[0][len(prefix):]
    
    prefix_text = " ".join([idx2word[t.item()] for t in prefix])
    pred_text = " ".join([idx2word[t.item()] for t in new_tokens])
    target_text = " ".join([idx2word[t.item()] for t in target])
    
    print(f"Префикс:    {prefix_text}")
    print(f"Предсказан: {pred_text}")
    print(f"Таргет:     {target_text}")
    print()


=== Примеры предсказаний LSTM ===

Префикс:    now i dont wanna face
Предсказан: and i have to be
Таргет:     my dad

Префикс:    getting ready to drive to the other side of denver to go quotcry in our <unk> with
Предсказан: the day to be a
Таргет:     a friend who closed her lss

Префикс:    for once it
Предсказан: was a good i have
Таргет:     wasnt

Префикс:    crud sites down again trying to get it back up waiting for host to reply
Предсказан: to be a good i
Таргет:     we shall be launching soon

Префикс:    follow if your
Предсказан: i have to be a
Таргет:     an aussie



Подсчет ROUGE для distilgpt2

In [12]:
from eval_transformer_pipeline import evaluate_transformer

transformer_results = evaluate_transformer(val_df["text"].tolist())
print(f"distilgpt2 ROUGE-1: {transformer_results['rouge1']:.4f}")
print(f"distilgpt2 ROUGE-2: {transformer_results['rouge2']:.4f}")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:5

distilgpt2 ROUGE-1: 0.0589
distilgpt2 ROUGE-2: 0.0188


Пример предсказания гпт

In [ ]:
print("=== Примеры предсказаний distilgpt2 ===\n")

samples = val_df["text"].tolist()[:10]

count = 0
for text in samples:
    if count >= 5:
        break
    words = text.split()
    if len(words) < 4:
        continue
    
    n = len(words)
    cut = int(n * 0.75)
    prefix = " ".join(words[:cut])
    target = " ".join(words[cut:])
    
    from eval_transformer_pipeline import generator
    result = generator(prefix, max_new_tokens=5, do_sample=False)
    generated_text = result[0]["generated_text"]
    pred_text = generated_text[len(prefix):].strip()
    
    print(f"Префикс:    {prefix}")
    print(f"Предсказан: {pred_text}")
    print(f"Таргет:     {target}")
    print()
    count += 1


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== Примеры предсказаний distilgpt2 ===



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Префикс:    now i dont wanna face my
Предсказан: own life.
Таргет:     dad ugh

Префикс:    getting ready to drive to the other side of denver to go quotcry in our drinksquot with a
Предсказан: little bit of a bit
Таргет:     friend who closed her lss today

Префикс:    for once it
Предсказан: was done.
Таргет:     wasnt mine



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Префикс:    crud sites down again trying to get it back up waiting for host to reply
Предсказан: .
Таргет:     we shall be launching soon maybe

Префикс:    follow if your an
Предсказан: cillary knowledge is not
Таргет:     aussie fan



Сравнение

In [13]:
import pandas as pd

comparison = pd.DataFrame({
    "Модель": ["LSTM (обученная)", "distilgpt2 (предобученная)"],
    "ROUGE-1": [lstm_results['rouge1'], transformer_results['rouge1']],
    "ROUGE-2": [lstm_results['rouge2'], transformer_results['rouge2']],
})
print(comparison.to_string(index=False))


                    Модель  ROUGE-1  ROUGE-2
          LSTM (обученная) 0.078262 0.008214
distilgpt2 (предобученная) 0.058925 0.018845


## Выводы

| Модель | ROUGE-1 | ROUGE-2 |
|--------|---------|---------|
| LSTM | 0.0778 | ... |
| distilgpt2 | 0.0702 | 0.0139 |

**LSTM** обучена на специфичных твитах и показывает сопоставимые метрики,
при этом весит значительно меньше (~2 МБ против ~350 МБ у distilgpt2).

**Рекомендация:** для мобильного приложения использовать LSTM — 
она удовлетворяет требованиям по памяти и скорости. 
distilgpt2 генерирует более связный текст, но слишком тяжёлая для смартфона.
